## Imports

In [9]:
import os
from pyspark.sql import SparkSession
os.environ["JAVA_HOME"] = "/opt/homebrew/opt/openjdk@17"

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("Bigdata") \
    .config("spark.sql.repl.eagerEval.enabled", "true") \
    .getOrCreate()

print("Spark session started ✅")

Spark session started ✅


In [10]:
base_path = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
base_path = os.path.join(base_path, "data")

parquets = ["orders", "customers", "order_items", "payments", "products", "sellers", "reviews"]

dataframes = {}
i = 0
for parquet in parquets:
    df = spark.read.parquet(f"{os.path.join(base_path, "2_silver", parquet)}")
    dataframes[parquet] = df
    i += 1
print(f"{i} dataframes successfully loaded ✅")

7 dataframes successfully loaded ✅


## Gold tables

### Sales KPIs (total and monthly CA, average cart, payments distribution)

In [11]:
from pyspark.sql import functions as F

df_sales = dataframes["orders"].filter(F.col("order_status") != "canceled") \
    .join(dataframes["payments"], on="order_id", how="inner") \
    .withColumn("month_year", F.date_format("order_purchase_timestamp", "yyyy-MM")) \
    .select("order_id", "customer_id", "payment_value", "payment_type", "month_year")


### Delivery and reviews KPIs (delivery delay, lateness rate, average rate)

In [12]:
df_deliveries_reviews = dataframes["orders"].filter(F.col("order_status") == "delivered") \
    .withColumn("delay_days", F.datediff("order_delivered_customer_date", "order_purchase_timestamp")) \
    .withColumn("is_late", F.when(F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"), True).otherwise(False)) \
    .select("order_id", "delay_days", "is_late") \
    .join(dataframes["reviews"], on="order_id", how="inner") \
    .select("order_id", "delay_days", "is_late", "review_score")


### Tops KPIs (top sellers, top categories)

In [13]:
df_products_sellers = dataframes["order_items"] \
    .join(dataframes["products"], on="product_id", how="inner") \
    .join(dataframes["sellers"], on="seller_id", how="inner") \
    .select("order_id", "product_id", "product_category_name_english", "price", "seller_id", "seller_city", "seller_state")

### Geo customers KPIs (to be joined to df_sales to get CA by city or state)

In [14]:
df_geo_customers = dataframes["orders"] \
    .join(dataframes["customers"], on="customer_id", how="inner") \
    .select("order_id", "customer_id", "customer_zip_code_prefix", "customer_city", "customer_state")

In [15]:
# Total revenue
total_ca = df_sales.agg(F.sum("payment_value").alias("total_revenue"))
print("=== TOTAL REVENUE ===")
total_ca.show()

# Monthly revenue
monthly_ca = df_sales.groupBy("month_year") \
    .agg(F.sum("payment_value").alias("monthly_revenue")) \
    .orderBy("month_year")
print("=== MONTHLY REVENUE ===")
monthly_ca.show()

# Average cart
avg_cart = df_sales.groupBy("order_id") \
    .agg(F.sum("payment_value").alias("order_total")) \
    .agg(F.avg("order_total").alias("average_cart"))
print("=== AVERAGE CART ===")
avg_cart.show()

# Payment distribution
payment_dist = df_sales.groupBy("payment_type") \
    .count() \
    .orderBy("count", ascending=False)
print("=== PAYMENT DISTRIBUTION ===")
payment_dist.show()

=== TOTAL REVENUE ===
+-------------+
|total_revenue|
+-------------+
|  15865616.52|
+-------------+

=== MONTHLY REVENUE ===
+----------+---------------+
|month_year|monthly_revenue|
+----------+---------------+
|   2016-09|         136.23|
|   2016-10|       53915.50|
|   2016-12|          19.62|
|   2017-01|      138119.76|
|   2017-02|      289081.01|
|   2017-03|      442406.37|
|   2017-04|      409846.01|
|   2017-05|      588529.96|
|   2017-06|      507302.62|
|   2017-07|      585331.36|
|   2017-08|      667224.47|
|   2017-09|      723781.27|
|   2017-10|      773104.35|
|   2017-11|     1187224.36|
|   2017-12|      874962.23|
|   2018-01|     1109464.00|
|   2018-02|      984790.19|
|   2018-03|     1156243.76|
|   2018-04|     1157336.33|
|   2018-05|     1149483.77|
+----------+---------------+
only showing top 20 rows
=== AVERAGE CART ===
+------------+
|average_cart|
+------------+
|  160.558787|
+------------+

=== PAYMENT DISTRIBUTION ===
+------------+-----+
|paym

In [18]:
# Annual revenue
annual_ca = df_sales.withColumn("year", F.year("month_year")) \
    .groupBy("year") \
    .agg(F.sum("payment_value").alias("annual_revenue")) \
    .orderBy("year")
print("=== ANNUAL REVENUE ===")
annual_ca.show()

=== ANNUAL REVENUE ===
+----+--------------+
|year|annual_revenue|
+----+--------------+
|2016|      54071.35|
|2017|    7186913.77|
|2018|    8624631.40|
+----+--------------+



In [17]:
# Average delivery delay
avg_delay = df_deliveries_reviews.agg(
    F.avg("delay_days").alias("avg_delivery_days")
)
print("=== AVERAGE DELIVERY DELAY ===")
avg_delay.show()

# Lateness rate
lateness_rate = df_deliveries_reviews.agg(
    (F.sum(F.col("is_late").cast("integer")) / F.count("*") * 100).alias("lateness_rate_%")
)
print("=== LATENESS RATE ===")
lateness_rate.show()

# Average review score
avg_score = df_deliveries_reviews.agg(
    F.avg("review_score").alias("avg_review_score")
)
print("=== AVERAGE REVIEW SCORE ===")
avg_score.show()

# Impact of lateness on satisfaction
impact = df_deliveries_reviews.groupBy("is_late") \
    .agg(F.avg("review_score").alias("avg_score")) \
    .orderBy("is_late")
print("=== IMPACT OF LATENESS ON SATISFACTION ===")
impact.show()

=== AVERAGE DELIVERY DELAY ===
+------------------+
| avg_delivery_days|
+------------------+
|12.461500939254616|
+------------------+

=== LATENESS RATE ===
+-----------------+
|  lateness_rate_%|
+-----------------+
|7.990784653542408|
+-----------------+

=== AVERAGE REVIEW SCORE ===
+-----------------+
| avg_review_score|
+-----------------+
|4.155716524320005|
+-----------------+

=== IMPACT OF LATENESS ON SATISFACTION ===
+-------+------------------+
|is_late|         avg_score|
+-------+------------------+
|  false| 4.293736817766549|
|   true|2.5664935064935066|
+-------+------------------+



In [19]:
# Annual revenue with percentage
total = df_sales.agg(F.sum("payment_value")).collect()[0][0]

annual_ca_pct = df_sales.withColumn("year", F.year("month_year")) \
    .groupBy("year") \
    .agg(F.sum("payment_value").alias("annual_revenue")) \
    .withColumn("percentage", F.round(F.col("annual_revenue") / total * 100, 2)) \
    .orderBy("year")

print("=== ANNUAL REVENUE WITH PERCENTAGE ===")
annual_ca_pct.show()

=== ANNUAL REVENUE WITH PERCENTAGE ===
+----+--------------+----------+
|year|annual_revenue|percentage|
+----+--------------+----------+
|2016|      54071.35|      0.34|
|2017|    7186913.77|     45.30|
|2018|    8624631.40|     54.36|
+----+--------------+----------+



In [20]:
# Payment distribution with percentage
total_payments = df_sales.count()

payment_dist_pct = df_sales.groupBy("payment_type") \
    .count() \
    .withColumn("percentage", F.round(F.col("count") / total_payments * 100, 2)) \
    .orderBy("count", ascending=False)

print("=== PAYMENT DISTRIBUTION WITH PERCENTAGE ===")
payment_dist_pct.show()

# Monthly revenue with percentage
monthly_ca_pct = df_sales.groupBy("month_year") \
    .agg(F.sum("payment_value").alias("monthly_revenue")) \
    .withColumn("percentage", F.round(F.col("monthly_revenue") / total * 100, 2)) \
    .orderBy("month_year")

print("=== MONTHLY REVENUE WITH PERCENTAGE ===")
monthly_ca_pct.show()

# Impact of lateness with percentage
total_orders = df_deliveries_reviews.count()
impact_pct = df_deliveries_reviews.groupBy("is_late") \
    .agg(
        F.count("*").alias("count"),
        F.avg("review_score").alias("avg_score")
    ) \
    .withColumn("percentage", F.round(F.col("count") / total_orders * 100, 2)) \
    .orderBy("is_late")

print("=== IMPACT OF LATENESS WITH PERCENTAGE ===")
impact_pct.show()

=== PAYMENT DISTRIBUTION WITH PERCENTAGE ===
+------------+-----+----------+
|payment_type|count|percentage|
+------------+-----+----------+
| credit_card|76351|     73.97|
|      boleto|19689|     19.07|
|     voucher| 5660|      5.48|
|  debit_card| 1522|      1.47|
+------------+-----+----------+

=== MONTHLY REVENUE WITH PERCENTAGE ===
+----------+---------------+----------+
|month_year|monthly_revenue|percentage|
+----------+---------------+----------+
|   2016-09|         136.23|      0.00|
|   2016-10|       53915.50|      0.34|
|   2016-12|          19.62|      0.00|
|   2017-01|      138119.76|      0.87|
|   2017-02|      289081.01|      1.82|
|   2017-03|      442406.37|      2.79|
|   2017-04|      409846.01|      2.58|
|   2017-05|      588529.96|      3.71|
|   2017-06|      507302.62|      3.20|
|   2017-07|      585331.36|      3.69|
|   2017-08|      667224.47|      4.21|
|   2017-09|      723781.27|      4.56|
|   2017-10|      773104.35|      4.87|
|   2017-11|     1

In [21]:
# Top 10 categories by revenue
top_categories = df_products_sellers \
    .groupBy("product_category_name_english") \
    .agg(F.sum("price").alias("revenue")) \
    .orderBy("revenue", ascending=False)

print("=== TOP 10 CATEGORIES ===")
top_categories.show(10)

# Top 10 sellers by revenue
top_sellers = df_products_sellers \
    .groupBy("seller_id", "seller_city", "seller_state") \
    .agg(F.sum("price").alias("revenue")) \
    .orderBy("revenue", ascending=False)

print("=== TOP 10 SELLERS ===")
top_sellers.show(10)

=== TOP 10 CATEGORIES ===
+-----------------------------+----------+
|product_category_name_english|   revenue|
+-----------------------------+----------+
|                health_beauty|1258681.34|
|                watches_gifts|1205005.68|
|               bed_bath_table|1036988.68|
|               sports_leisure| 988048.97|
|         computers_accesso...| 911954.32|
|              furniture_decor| 729762.49|
|                   cool_stuff| 635290.85|
|                   housewares| 632248.66|
|                         auto| 592720.11|
|                 garden_tools| 485256.46|
+-----------------------------+----------+
only showing top 10 rows
=== TOP 10 SELLERS ===
+--------------------+----------------+------------+---------+
|           seller_id|     seller_city|seller_state|  revenue|
+--------------------+----------------+------------+---------+
|4869f7a5dfa277a7d...|         guariba|          SP|229472.63|
|53243585a1d6dc264...|lauro de freitas|          BA|222776.05|
|4a3ca931

In [22]:
# Total revenue for percentage calculation
total_product_revenue = df_products_sellers.agg(F.sum("price")).collect()[0][0]

# Top categories with percentage
top_categories = df_products_sellers \
    .groupBy("product_category_name_english") \
    .agg(F.sum("price").alias("revenue")) \
    .withColumn("percentage", F.round(F.col("revenue") / total_product_revenue * 100, 2)) \
    .orderBy("revenue", ascending=False)

# Top sellers with percentage
top_sellers = df_products_sellers \
    .groupBy("seller_id", "seller_city", "seller_state") \
    .agg(F.sum("price").alias("revenue")) \
    .withColumn("percentage", F.round(F.col("revenue") / total_product_revenue * 100, 2)) \
    .orderBy("revenue", ascending=False)

print("=== TOP 10 CATEGORIES WITH PERCENTAGE ===")
top_categories.show(10)

print("=== TOP 10 SELLERS WITH PERCENTAGE ===")
top_sellers.show(10)

=== TOP 10 CATEGORIES WITH PERCENTAGE ===
+-----------------------------+----------+----------+
|product_category_name_english|   revenue|percentage|
+-----------------------------+----------+----------+
|                health_beauty|1258681.34|      9.26|
|                watches_gifts|1205005.68|      8.87|
|               bed_bath_table|1036988.68|      7.63|
|               sports_leisure| 988048.97|      7.27|
|         computers_accesso...| 911954.32|      6.71|
|              furniture_decor| 729762.49|      5.37|
|                   cool_stuff| 635290.85|      4.67|
|                   housewares| 632248.66|      4.65|
|                         auto| 592720.11|      4.36|
|                 garden_tools| 485256.46|      3.57|
+-----------------------------+----------+----------+
only showing top 10 rows
=== TOP 10 SELLERS WITH PERCENTAGE ===
+--------------------+----------------+------------+---------+----------+
|           seller_id|     seller_city|seller_state|  revenue|pe

In [23]:
avg_cart = df_sales.groupBy("order_id") \
    .agg(F.sum("payment_value").alias("order_total")) \
    .agg(F.avg("order_total").alias("average_cart"))

print("=== AVERAGE CART ===")
avg_cart.show()

=== AVERAGE CART ===
+------------+
|average_cart|
+------------+
|  160.558787|
+------------+



## Exports

In [16]:
gold_dataframes = {
    "sales": df_sales,
    "deliveries_reviews": df_deliveries_reviews,
    "products_sellers": df_products_sellers,
    "geo_customers": df_geo_customers
}

for name, df in gold_dataframes.items():
    df.write.mode("overwrite").parquet(f"{base_path}/3_gold/{name}")
    print(f"Dataframe {name} saved to gold ✅")

Dataframe sales saved to gold ✅
Dataframe deliveries_reviews saved to gold ✅
Dataframe products_sellers saved to gold ✅
Dataframe geo_customers saved to gold ✅
